# V1 MRI → báo cáo

Notebook private. Xem `docs/README_KAGGLE.md` trong repo trước khi chạy. Add Input các dataset private; sao chép đúng đường dẫn và version vào cell tham số. V2 template chạy CPU, V1/V2 LoRA cần GPU. Không dùng output template làm nhãn bác sĩ.

Chạy smoke trước; đổi `mode` thành `full` và đặt `run_name` mới cho run chính. Sau khi dừng, xác nhận checkpoint nằm trong Output của Saved Version.

In [ ]:
CODE_REF = "codex/kaggle-v1-v2"  # đổi thành commit SHA để khóa thí nghiệm
REPO_URL = "https://github.com/kttt294/MRI-report-generator.git"

In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    raise RuntimeError("Repo đã tồn tại: restart session sạch hoặc dùng checkout hiện tại có kiểm soát.")
subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", CODE_REF], cwd=REPO, check=True)
CODE_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Code commit:", CODE_SHA)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-kaggle.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable, "scripts/check_environment.py"], cwd=REPO, check=True)

In [ ]:
CONFIG = {
    "task": "v1-train", "mode": "smoke", "run_name": "v1-train-smoke-01",
    "annotations_root": "/kaggle/input/CHANGE-ME-annotations",
    "annotations_dataset": "OWNER/SLUG/VERSION",
    "images_root": "/kaggle/input/CHANGE-ME-images/nifti",
    "images_dataset": "OWNER/SLUG/VERSION",
    "work_root": "/kaggle/working", "cache_root": "/tmp/mri-cache",
    "fold": 1, "gpu_index": "0", "max_runtime_minutes": 30,
    "resume_from": None,
    "reviewed_targets": None,
    "report_overrides": {"backend": "template"},
}

In [ ]:
import json
CONFIG["code_commit"] = CODE_SHA
config_path = Path("/kaggle/working/cloud_run.json")
config_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
subprocess.run([sys.executable, "scripts/cloud_run.py", "--config", str(config_path)], cwd=REPO, check=True)

In [ ]:
run_dir = Path(CONFIG["work_root"]) / "runs" / CONFIG["run_name"]
print("Kết quả:", run_dir)
print("\n".join(str(p.relative_to(run_dir)) for p in sorted(run_dir.iterdir())))
print("Save Version → kiểm tra Output; final_adapter để inference, checkpoint-* có COMPLETE.json để resume.")

## Chạy tiếp / inference

V1: sau train, tạo run mới với `task='v1-infer'`, thêm `adapter_path` trỏ đến `final_adapter`, và `split='val'` khi đang phát triển. Chỉ dùng test sau khi khóa thiết kế.

Resume: Add Input outputs của Saved Version cũ, đặt `resume_from` tới `checkpoint-N` có `COMPLETE.json`; giữ code SHA, model/config, fold, mode và data versions như cũ. Đổi `run_name`, giữ thời gian dự phòng để lưu outputs. Smoke không được resume thành full.

V2 LLM: đặt `report_overrides` theo README; kiểm tra cả attempts và fallback. V2 train cần file targets có duyệt thật, không đổi `pending` hàng loạt thành `accepted`.